In [2]:
from dragonfly import Window
from dragonfly.windows.rectangle   import Rectangle
from dragonfly.windows.darwin_window import DarwinWindow
# from dragonfly.windows.win32_window import Win32Window
from dataclasses import dataclass
import time

In [3]:
@dataclass
class WindowInfo:
    title: str
    executable: str
    handle: int
    window: DarwinWindow # assume we only handle macOS and Windows

In [4]:
def is_window_important(window: DarwinWindow) -> bool:
    return True
    # ignore windows without a title or executable because these are usually system processes
    if not window.title or not window.executable:
        return False
    
    if not isinstance(window, DarwinWindow):
        # ignore windows that are not enabled (cannot receive mouse/keyboard input)
        if not window.is_enabled:
            return False
        
        # ignore windows that are not visible because they are usually not user facing apps
        if not window.is_visible:
            return False

        # ignore windows that are part of the OS
        # user applications are usually in C:\Program Files\ or /Applications
        if window.executable.startswith("C:\\Windows\\"):
            return False
        
    
    elif isinstance(window, DarwinWindow):
        # TODO: write macOS filtering logic as needed
        pass # do nothing for now
    
    return True

In [5]:
def get_window_list() -> list[WindowInfo]:
    windows = DarwinWindow.get_all_windows()
    window_info: list[WindowInfo] = []

    for window in windows:
        if not is_window_important(window):
            continue
        try:
            info = {
                'title': window.title,
                'executable': window.executable,
                'handle': window.handle,
                'window': window
            }
            window_info.append(info)
        except Exception as e:
            print(f"Error getting info for window: {e}")

    return window_info

In [6]:
window_list = get_window_list()
for window in window_list:
    print(f"Window: {window['title']}")
    print(f"  Executable: {window['executable']}")
    print(f"  Handle: {window['handle']}")
    # print(f"  is_visible: {window['window'].is_visible} | is_minimized: {window['window'].is_minimized} | is_maximized: {window['window'].is_maximized}")
    print("-" * 50)

    

Window: 
  Executable: Google Chrome
  Handle: 172074
--------------------------------------------------
Window: Accessibility
  Executable: System Settings
  Handle: 184365
--------------------------------------------------
Window: Notes
  Executable: Notes
  Handle: 196656
--------------------------------------------------
Window: testing.ipynb — clappy
  Executable: Electron
  Handle: 200753
--------------------------------------------------
Window: Activity Monitor – My Processes
  Executable: Activity Monitor
  Handle: 204850
--------------------------------------------------
Window: filtered.txt
  Executable: TextEdit
  Handle: 208947
--------------------------------------------------
Window: -zsh
  Executable: iTerm2
  Handle: 213044
--------------------------------------------------
Window: hannahguo — -zsh — 80×24
  Executable: Terminal
  Handle: 217141
--------------------------------------------------
Window: bin
  Executable: Finder
  Handle: 233529
------------------------

In [10]:
window_shake = window_list[10]['window']
print(window_shake.executable)

Discord


In [8]:
import math, random

In [18]:
print(window_shake.minimize())

True


In [1]:
import applescript
script = f'''
tell application "System Events"
    set targetProcess to (first process whose unix id is 1810)
    # set windowProperties to {{}}
    repeat with w in windows of targetProcess
        set end of windowProperties to {{name of w, position of w, size of w, id of w}}
    end repeat
end tell
return windowProperties
'''
properties = applescript.AppleScript(script).run()
print(properties)

ScriptError: System Events got an error: Python is not allowed assistive access. (-25211) app='System Events' range=134-276

In [17]:
print(window_shake.get_properties())

{'minW': 'msng', 'orie': 'msng', 'posn': [247, 247], 'axds': 'msng', 'rold': 'standard window', 'focu': False, 'titl': 'University Of Waterloo | University Of Waterloo - Discord', 'ptsz': [1463, 860], 'help': 'msng', 'ects': [], 'enaB': 'msng', 'maxV': 'msng', 'role': 'AXWindow', 'valL': 'msng', 'sbrl': 'AXStandardWindow', 'selE': 'msng', 'pnam': 'University Of Waterloo | University Of Waterloo - Discord', 'desc': 'standard window', 'pcls': 'cwin'}


In [16]:
# using the starting position, execute a series of window movements that make the window appear like it is shaking for a second
starting_position = window_shake.get_position()
start_l, start_t, start_w, start_h = starting_position.ltwh

if window_shake.is_maximized:
    window_shake.restore()
    window_shake.set_position(Rectangle(0, 0, start_w, start_h))
    time.sleep(0.3)


shake_iterations = 20
intensity = 20
duration = 0.4
interval = duration / shake_iterations

end_time = time.time() + duration

# Repeatedly update the window's position.
while time.time() < end_time:
    # Calculate a random offset (you can add damping if you want a decaying shake).
    dx = random.randint(-intensity, intensity)
    dy = random.randint(-intensity, intensity)

    # Update the window's position.
    window_shake.set_position(Rectangle(start_l + dx, start_t + dy,
                        start_w + dx, start_h + dy))

    time.sleep(interval)

window_shake.set_position(starting_position)
window_shake.maximize()

True

In [152]:
window_shake.get_position()

Rectangle(-10.0, 0.0, 1926.0, 1128.0)

In [155]:
window_shake.get_position()

Rectangle(229.0, 75.0, 1440.0, 990.0)